# Analog Selection Refinement

Sixth alignment-track experiment. Earlier screens showed that nearest-analog transfer has real but sub-gate signal. This notebook tests whether changing the analog selection geometry, especially toward log-shape and typewell-alignment descriptors, improves fold-safe analog transfer.

## 1. Setup

In [1]:
"Configure imports, paths, and analog-selection candidate families."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__analog_selection_refinement"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
DESCRIPTOR_PATH = ARTIFACT_TABLES / "model__typewell_alignment_feature_descriptors__2026-06-23.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0

SCHEDULES = {
    "fixed_s010": {"z0": 0.10, "z1": 0.10, "z2": 0.10, "z3": 0.10},
    "progress_gate": {"z0": 0.025, "z1": 0.025, "z2": 0.075, "z3": 0.10},
    "late_only": {"z0": 0.00, "z1": 0.00, "z2": 0.10, "z3": 0.10},
}

FEATURE_SETS = {
    "baseline_hybrid": ["known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300", "typewell_corr_last300", "typewell_best_lag_corr", "typewell_best_lag_tvt", "xy_span_known", "z_drop_known"],
    "log_shape": ["known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "typewell_best_lag_tvt", "typewell_best_lag_mae"],
    "typewell_quality": ["typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "typewell_best_lag_tvt", "typewell_best_lag_mae"],
    "trajectory_depth": ["known_tvt_span", "known_md_span", "xy_span_known", "z_drop_known", "last_known_tvt"],
    "late_extrapolation": ["last_known_tvt", "known_tvt_span", "known_md_span", "known_gr_slope_last300", "typewell_best_lag_corr", "typewell_best_lag_tvt", "xy_span_known"],
    "log_plus_trajectory": ["known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300", "typewell_best_lag_corr", "typewell_best_lag_tvt", "known_tvt_span", "xy_span_known", "z_drop_known"],
}

CANDIDATES = [{"name": "baseline_no_refined_analog", "feature_set": "none", "k": 0, "schedule": "none"}]
for feature_set in FEATURE_SETS:
    for k in [1, 2, 3]:
        for schedule in SCHEDULES:
            CANDIDATES.append({"name": f"{feature_set}_k{k}_{schedule}", "feature_set": feature_set, "k": k, "schedule": schedule})

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"candidate_count={len(CANDIDATES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__analog_selection_refinement
candidate_count=55


## 2. Load Residuals And Descriptors

In [2]:
"Load current residuals and descriptor features used for analog selection."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "eval_progress", "y_true", "y_pred", "residual"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["fold"] = residuals["fold"].astype(int)
residuals["row_idx"] = residuals["row_idx"].astype(int)
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]
residuals["pseudo_zone"] = pd.cut(residuals["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])

descriptors = pd.read_csv(DESCRIPTOR_PATH)
descriptor_columns = list(dict.fromkeys(["wellname", "last_known_tvt"] + sorted(set(sum(FEATURE_SETS.values(), [])))))
model_frame = residuals.merge(descriptors[descriptor_columns], on="wellname", how="left")
model_frame["baseline_delta_from_last_known"] = model_frame["baseline_pred"] - model_frame["last_known_tvt"]
model_frame["true_delta_from_last_known"] = model_frame["y_true"] - model_frame["last_known_tvt"]

print(f"rows={len(model_frame):,}")
print(f"wells={model_frame['wellname'].nunique():,}")
print(model_frame.groupby("pseudo_zone", observed=False).size().rename("rows_by_zone").to_string())

rows=5,400
wells=773
pseudo_zone
z0    1160
z1    1414
z2    1412
z3    1414


## 3. Fold-Safe Analog Selection Functions

In [3]:
"Define fold-safe nearest-analog selection and target-zone delta transfer."


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def scaled_descriptor_matrices(train_desc, query_desc, features):
    med = train_desc[features].median(numeric_only=True)
    train_values = train_desc[features].fillna(med).to_numpy(dtype=float)
    query_values = query_desc[features].fillna(med).to_numpy(dtype=float)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_values)
    query_scaled = scaler.transform(query_values)
    return train_scaled, query_scaled


def interpolate_delta(train_rows, analog_well, progress_values):
    curve = train_rows.loc[train_rows["wellname"].eq(analog_well), ["eval_progress", "true_delta_from_last_known"]].dropna().sort_values("eval_progress")
    if len(curve) < 2:
        return np.full(len(progress_values), np.nan)
    grouped = curve.groupby("eval_progress", as_index=False)["true_delta_from_last_known"].mean()
    return np.interp(np.asarray(progress_values, dtype=float), grouped["eval_progress"].to_numpy(), grouped["true_delta_from_last_known"].to_numpy(), left=grouped["true_delta_from_last_known"].iloc[0], right=grouped["true_delta_from_last_known"].iloc[-1])


def weighted_delta(train_rows, train_names, distances, progress_values, k):
    order = np.argsort(distances)[:k]
    picked_names = train_names[order]
    picked_distances = distances[order]
    weights = 1.0 / np.maximum(picked_distances, 1e-6)
    weights = weights / weights.sum()
    curves = []
    for analog_well in picked_names:
        curves.append(interpolate_delta(train_rows, analog_well, progress_values))
    curve_matrix = np.vstack(curves)
    return np.nansum(curve_matrix * weights[:, None], axis=0), ";".join(map(str, picked_names)), float(np.mean(picked_distances))


def schedule_values(frame, schedule_name):
    schedule = SCHEDULES[schedule_name]
    return frame["pseudo_zone"].astype(str).map(schedule).fillna(0.0).to_numpy(dtype=float)

print("ready")

ready


## 4. Out-Of-Fold Selection Evaluation

In [4]:
"Evaluate analog-selection feature sets with fixed and progress-gated schedules."

start_time = time.time()
metric_rows = []
prediction_rows = []
selection_rows = []
for candidate in CANDIDATES:
    for fold in sorted(model_frame["fold"].unique()):
        train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
        valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
        analog_gap = np.zeros(len(valid_rows), dtype=float)
        analog_names = np.array([""] * len(valid_rows), dtype=object)
        analog_distance = np.zeros(len(valid_rows), dtype=float)
        if candidate["feature_set"] != "none":
            features = FEATURE_SETS[candidate["feature_set"]]
            train_desc = descriptors.loc[descriptors["wellname"].isin(train_rows["wellname"].unique())].copy().reset_index(drop=True)
            valid_desc = descriptors.loc[descriptors["wellname"].isin(valid_rows["wellname"].unique())].copy().reset_index(drop=True)
            train_scaled, valid_scaled = scaled_descriptor_matrices(train_desc, valid_desc, features)
            train_names = train_desc["wellname"].to_numpy()
            valid_lookup = {name: i for i, name in enumerate(valid_desc["wellname"].to_numpy())}
            for wellname, rows in valid_rows.groupby("wellname"):
                q_idx = valid_lookup[wellname]
                distances = np.sqrt(((train_scaled - valid_scaled[q_idx]) ** 2).mean(axis=1))
                delta_pred, selected_names, mean_distance = weighted_delta(train_rows, train_names, distances, rows["eval_progress"].to_numpy(), candidate["k"])
                row_idx = rows.index.to_numpy()
                analog_gap[np.searchsorted(valid_rows.index.to_numpy(), row_idx)] = delta_pred - rows["baseline_delta_from_last_known"].to_numpy()
                analog_names[np.searchsorted(valid_rows.index.to_numpy(), row_idx)] = selected_names
                analog_distance[np.searchsorted(valid_rows.index.to_numpy(), row_idx)] = mean_distance
                selection_rows.append({"candidate": candidate["name"], "fold": int(fold), "wellname": wellname, "selected_analogs": selected_names, "mean_distance": mean_distance})
        if candidate["schedule"] == "none":
            shrink = np.zeros(len(valid_rows), dtype=float)
        else:
            shrink = schedule_values(valid_rows, candidate["schedule"])
        correction = np.clip(analog_gap * shrink, -OFFSET_CLIP, OFFSET_CLIP)
        y_true = valid_rows["y_true"].to_numpy()
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        metric_rows.append(
            {
                "candidate": candidate["name"],
                "feature_set": candidate["feature_set"],
                "k": int(candidate["k"]),
                "schedule": candidate["schedule"],
                "fold": int(fold),
                "rmse": rmse(y_true, y_pred),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "pseudo_zone", "y_true", "baseline_pred", "baseline_residual", "eval_progress"]].copy()
        out["candidate"] = candidate["name"]
        out["feature_set"] = candidate["feature_set"]
        out["selected_analogs"] = analog_names
        out["analog_distance"] = analog_distance
        out["analog_gap"] = analog_gap
        out["analog_shrink"] = shrink
        out["analog_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_selection"] = out["y_true"] - out["y_pred"]
        prediction_rows.append(out)

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
selection_diagnostics = pd.DataFrame(selection_rows)
summary = (
    fold_metrics.groupby(["candidate", "feature_set", "k", "schedule"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_refined_analog"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.head(20).to_string(index=False))

                           candidate         feature_set  k      schedule  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
log_plus_trajectory_k1_progress_gate log_plus_trajectory  1 progress_gate  14.638289  0.411922 10.353734        -0.043459        1.207596            8.494656    5400      773                      -0.100256
   log_plus_trajectory_k1_fixed_s010 log_plus_trajectory  1    fixed_s010  14.639849  0.394680 10.370036        -0.067951        1.614807            8.494656    5400      773                      -0.098696
    log_plus_trajectory_k1_late_only log_plus_trajectory  1     late_only  14.639864  0.410843 10.361632        -0.043515        1.321253            8.494656    5400      773                      -0.098681
    baseline_hybrid_k1_progress_gate     baseline_hybrid  1 progress_gate  14.640240  0.350077 10.364496         0.001671        1.228527            9.375205    5400      773  

## 5. Diagnostics And Export

In [5]:
"Summarize the best selection geometry and export artifacts."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_refined_analog")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
zone_summary = (
    best_predictions.groupby("pseudo_zone", observed=False)
    .agg(
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        rmse_after=("residual_after_selection", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_shrink=("analog_shrink", "mean"),
        rows=("row_idx", "size"),
    )
    .reset_index()
)
zone_summary["rmse_delta"] = zone_summary["rmse_after"] - zone_summary["rmse_before"]

well_before = model_frame.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_selection", "mean"), mean_analog_correction=("analog_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()
family_summary = summary.groupby("feature_set", as_index=False).agg(best_rmse=("rmse_mean", "min"), best_delta=("rmse_delta_vs_sample_baseline", "min")).sort_values("best_rmse")

findings = pd.DataFrame(
    [
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}", "modeling_implication": "Promote selection refinement only if it clears the 0.05 RMSE gate or gives a strong Kaggle-safe hypothesis."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "Compare against prior nearest-analog fixed/gated results around -0.042 RMSE."},
        {"finding": "best_feature_set", "value": f"{best_row['feature_set']} with k={int(best_row['k'])} and schedule={best_row['schedule']}", "modeling_implication": "Identifies the analog-selection geometry worth carrying forward."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive means selection adds well-level noise."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_well_diagnostics__{DATE_STAMP}.csv"
zone_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_zone_summary__{DATE_STAMP}.csv"
selection_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_selection_diagnostics__{DATE_STAMP}.csv"
family_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_family_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__analog_selection_refinement_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__analog_selection_refinement__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
zone_summary.to_csv(zone_path, index=False)
selection_diagnostics.to_csv(selection_path, index=False)
family_summary.to_csv(family_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Analog Selection Refinement

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Best feature set: `{best_row['feature_set']}`
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests whether analog transfer improves when nearest analogs are selected by different fold-safe descriptor geometries. It is a residual-sample diagnostic, not a Kaggle submission notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_analog_selection",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(FEATURE_SETS.get(best_row["feature_set"], []))),
            "notes": "Fold-safe analog selection geometry screen over current Huber + additive Hilbert residual sample",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, zone_path, selection_path, family_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "offset_clip": OFFSET_CLIP, "schedules": SCHEDULES, "feature_sets": FEATURE_SETS, "candidates": CANDIDATES, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(family_summary.to_string(index=False))
print(zone_summary.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                                        value                                                                                        modeling_implication
       best_candidate log_plus_trajectory_k1_progress_gate RMSE 14.6383 vs sample baseline 14.7385 Promote selection refinement only if it clears the 0.05 RMSE gate or gives a strong Kaggle-safe hypothesis.
sample_baseline_delta                                                 best candidate delta -0.1003                                Compare against prior nearest-analog fixed/gated results around -0.042 RMSE.
     best_feature_set                      log_plus_trajectory with k=1 and schedule=progress_gate                                            Identifies the analog-selection geometry worth carrying forward.
     well_bias_change                                      median absolute well-bias delta -0.0963                                           Negative is good; positive mean

## 6. Conclusion

In [6]:
"Print final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=log_plus_trajectory_k1_progress_gate
sample_baseline_rmse=14.7385
best_rmse=14.6383
delta_vs_sample_baseline=-0.1003
promotion_gate_cleared=True
